# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
md = ds.metadata

# Print core metadata
print("Dataset: {}".format(md.name))
print("Description: {}".format(md.description))
print("Version: {}".format(md.version if hasattr(md, 'version') else 'N/A'))
print("Identifier: {}".format(md.identifier if hasattr(md, 'identifier') else 'N/A'))

## 2. Data Overview
Review available record sets and their `@id`s, as well as fields within each record set.

This helps you understand what tabular data and attributes are present.

In [ ]:
# List all record sets by their @id
print("Available record sets:")
record_set_ids = [rs['@id'] for rs in md.recordSet] if hasattr(md, 'recordSet') and md.recordSet else []
for rs in md.recordSet:
    rs_id = rs['@id']
    rs_name = rs.get('name', 'N/A')
    print(f"- @id: {rs_id}   Name: {rs_name}")
    # List fields in this record set
    field_list = rs.get('field', [])
    if isinstance(field_list, dict):
        field_list = [field_list]
    print("  Fields (by @id and name):")
    for fld in field_list:
        field_id = fld['@id']
        field_name = fld.get('name', 'N/A')
        print(f"    - {field_id}: {field_name}")
    print()
if not record_set_ids:
    print("No record sets defined in this Croissant. Please check the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 

All fields and record sets are referenced by their `@id` as per the Croissant schema.

In [ ]:
dataframes = {}

if record_set_ids:
    # For demonstration, use the first record set @id
    for rid in record_set_ids:
        print(f"Loading records for record set {rid} ...")
        # Returns a generator of dicts, each dict is a record
        recs = list(ds.records(record_set=rid))
        if len(recs) > 0:
            df = pd.DataFrame(recs)
            dataframes[rid] = df
            print(f"Fields (columns) for record set {rid}:")
            print(df.columns.tolist())
            print(df.head())
        else:
            print(f"No records found for record set {rid}.")
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering and normalizing a numeric field. 

All entities are referenced by their `@id`. Adjust the field/column names as appropriate for the actual dataset.

In [ ]:
# Example: pick first tabular record set and attempt EDA on a representative numeric field
if dataframes:
    rec_id = list(dataframes.keys())[0]
    df = dataframes[rec_id]
    print(f"Using record set: {rec_id}")
    print(f"Available fields: {df.columns.tolist()}")

    # Guess a numeric field by searching for typical mass/protein fields
    numeric_candidates = [
        c for c in df.columns if (
            "MW" in c or "mass" in c.lower() or "peptide" in c.lower() or "coverage" in c.lower() or "abundance" in c.lower() or "count" in c.lower()
        )
    ]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Example numeric field chosen: {numeric_field}")
        # Clean to numeric as needed
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        # Filtering
        threshold = df[numeric_field].mean() if df[numeric_field].mean() is not None else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} row(s)")
        print(filtered_df.head())

        # Normalization
        norm_col = numeric_field + "_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, norm_col]].head())

        # Look for a groupable field (e.g. 'sample', 'condition', 'id')
        group_candidates = [c for c in df.columns if c.lower() in ["sample", "group", "condition"]]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No obvious grouping field found.")
    else:
        print("No numeric fields found suitable for EDA in the record set.")
else:
    print("No loaded dataframes to perform EDA.")

## 5. Visualization
Visualize the distribution of a numeric field or relationship between fields. This requires `matplotlib` or `seaborn`, which can be installed if not present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    rec_id = list(dataframes.keys())[0]
    df = dataframes[rec_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8, 4))
        df[numeric_field].dropna().hist(bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("Unable to plot: No numeric field detected.")
else:
    print("No dataframes found for plotting.")

## 6. Conclusion
This notebook demonstrated how to load metadata and records from a Croissant dataset using `mlcroissant`, and provided an initial exploratory analysis using the dataset's record sets, fields, and columns referenced by their `@id`s.

You may now proceed to perform advanced analyses, visualize specific features, or integrate this data into ML pipelines using pandas and other libraries.